# FIFA World Cup 2026 Prediction Model

**Business Question:** Which team is most likely to win the 2026 FIFA World Cup?

**Approach:**
1. Build a custom Elo rating system from 150+ years of international results
2. Train a match outcome model on Elo rating differences
3. Simulate the 2026 tournament 10,000 times to estimate win probabilities

**Data:** International football results 1872–present ([source](https://github.com/martj42/international_results))

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Load Data

In [ ]:
# Load international football results (1872-present)
url = 'https://raw.githubusercontent.com/martj42/international_results/master/results.csv'
df = pd.read_csv(url, parse_dates=['date'])

print(f"Total matches: {len(df):,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique teams: {pd.concat([df['home_team'], df['away_team']]).nunique()}")
df.head()

In [ ]:
# Tournament types in the data
print(df['tournament'].value_counts().head(20))

## 2. Elo Rating System

Elo ratings measure team strength on a single scale. After each match:
- The winning team gains points; the losing team loses points
- The amount transferred depends on how surprising the result was
- Important matches (World Cup) have a higher K-factor than friendlies

In [ ]:
# K-factor by tournament importance
def get_k_factor(tournament):
    t = tournament.lower()
    if 'fifa world cup' in t and 'qualification' not in t:
        return 60
    elif 'confederation' in t or 'continental' in t or 'copa america' in t or 'euro' in t or 'africa cup' in t or 'gold cup' in t or 'asian cup' in t:
        return 50
    elif 'qualification' in t or 'qualifier' in t:
        return 40
    elif 'friendly' in t:
        return 20
    else:
        return 35

# Goal difference multiplier
def goal_diff_multiplier(goal_diff):
    if goal_diff == 1:
        return 1.0
    elif goal_diff == 2:
        return 1.5
    elif goal_diff == 3:
        return 1.75
    else:
        return 1.75 + (goal_diff - 3) * 0.05

# Expected result given Elo difference
def expected_result(elo_a, elo_b, home_advantage=100):
    return 1 / (1 + 10 ** (-(elo_a + home_advantage - elo_b) / 400))

print("Elo functions defined.")

In [ ]:
# Compute Elo ratings for all teams across all matches
elo_ratings = {}  # team -> current elo
match_elos = []   # store elo at time of each match for model training

for _, row in df.iterrows():
    home = row['home_team']
    away = row['away_team']
    neutral = row['neutral']

    # Initialize new teams at 1500
    if home not in elo_ratings:
        elo_ratings[home] = 1500
    if away not in elo_ratings:
        elo_ratings[away] = 1500

    home_elo = elo_ratings[home]
    away_elo = elo_ratings[away]

    # No home advantage for neutral venues
    home_adv = 0 if neutral else 100
    expected_home = expected_result(home_elo, away_elo, home_adv)

    # Actual result: 1 = home win, 0.5 = draw, 0 = away win
    if row['home_score'] > row['away_score']:
        actual = 1.0
    elif row['home_score'] == row['away_score']:
        actual = 0.5
    else:
        actual = 0.0

    k = get_k_factor(row['tournament'])
    gd = abs(row['home_score'] - row['away_score'])
    gdm = goal_diff_multiplier(gd)

    # Elo update
    delta = k * gdm * (actual - expected_home)
    elo_ratings[home] += delta
    elo_ratings[away] -= delta

    # Store pre-match elos for model training
    match_elos.append({
        'date': row['date'],
        'home_team': home,
        'away_team': away,
        'tournament': row['tournament'],
        'neutral': neutral,
        'home_elo_pre': home_elo,
        'away_elo_pre': away_elo,
        'elo_diff': home_elo - away_elo,
        'home_score': row['home_score'],
        'away_score': row['away_score'],
        'result': actual
    })

match_df = pd.DataFrame(match_elos)
print(f"Matches processed: {len(match_df):,}")
print(f"\nTop 20 teams by current Elo rating:")
top_teams = pd.Series(elo_ratings).sort_values(ascending=False).head(20)
print(top_teams.round(0).astype(int).to_string())

In [ ]:
# Visualize top 20 teams by Elo
fig, ax = plt.subplots(figsize=(10, 6))
top20 = pd.Series(elo_ratings).sort_values(ascending=False).head(20)
bars = ax.barh(top20.index[::-1], top20.values[::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Elo Rating')
ax.set_title('Top 20 International Teams by Elo Rating', fontsize=14)
ax.axvline(1500, color='gray', linestyle='--', alpha=0.5, label='Average (1500)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Match Outcome Model

Using Elo difference as the primary predictor for match outcome. This gives us calibrated win probabilities for any matchup.

In [ ]:
# Focus on decisive matches (no draws) for binary classification
# Filter to competitive matches from 1970 onward (modern football era)
model_df = match_df[
    (match_df['date'].dt.year >= 1970) &
    (match_df['result'] != 0.5)  # exclude draws for binary model
].copy()

model_df['home_win'] = (model_df['result'] == 1.0).astype(int)
model_df['home_adv'] = (~model_df['neutral']).astype(int)

features = ['elo_diff', 'home_adv']
X = model_df[features]
y = model_df['home_win']

# Time-based split: train on matches before 2018, test on 2018 onward
split_date = pd.Timestamp('2018-01-01')
X_train = X[model_df['date'] < split_date]
X_test = X[model_df['date'] >= split_date]
y_train = y[model_df['date'] < split_date]
y_test = y[model_df['date'] >= split_date]

print(f"Train: {len(X_train):,} matches | Test: {len(X_test):,} matches")

In [ ]:
# Train logistic regression
model = LogisticRegression()
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)
brier = brier_score_loss(y_test, y_proba)

print(f"ROC-AUC:     {auc:.3f}")
print(f"Brier Score: {brier:.3f}  (lower is better, 0.25 = random)")

## 4. 2026 World Cup Simulation

Monte Carlo simulation: run the tournament 10,000 times using current Elo ratings. Each match is simulated by drawing from the predicted win probabilities.

In [ ]:
# 2026 World Cup qualified teams (32 strongest by Elo as proxy)
# Using top Elo-rated teams that typically qualify
wc_2026_teams = [
    'Brazil', 'France', 'England', 'Argentina', 'Spain', 'Portugal',
    'Netherlands', 'Germany', 'Belgium', 'Italy', 'Croatia', 'Uruguay',
    'Colombia', 'Mexico', 'United States', 'Canada', 'Morocco', 'Senegal',
    'Japan', 'South Korea', 'Australia', 'Ecuador', 'Switzerland', 'Denmark',
    'Poland', 'Serbia', 'Hungary', 'Cameroon', 'Nigeria', 'Ghana',
    'Saudi Arabia', 'Iran'
]

# Get current Elo for each team
team_elos = {team: elo_ratings.get(team, 1500) for team in wc_2026_teams}

print("2026 World Cup teams and current Elo ratings:")
for team, elo in sorted(team_elos.items(), key=lambda x: -x[1]):
    print(f"  {team:<25} {int(elo)}")

In [ ]:
def simulate_match(team_a, team_b, elos, neutral=True):
    """Simulate a single match. Returns winner."""
    elo_a = elos[team_a]
    elo_b = elos[team_b]
    home_adv = 0  # neutral ground for World Cup
    elo_diff = elo_a - elo_b
    win_prob = model.predict_proba([[elo_diff, home_adv]])[0][1]
    return team_a if np.random.random() < win_prob else team_b


def simulate_tournament(teams, elos):
    """Simulate a 32-team knockout tournament."""
    remaining = teams.copy()
    np.random.shuffle(remaining)

    while len(remaining) > 1:
        next_round = []
        for i in range(0, len(remaining), 2):
            if i + 1 < len(remaining):
                winner = simulate_match(remaining[i], remaining[i+1], elos)
                next_round.append(winner)
            else:
                next_round.append(remaining[i])  # bye
        remaining = next_round

    return remaining[0]


# Run 10,000 simulations
N_SIMS = 10_000
win_counts = {team: 0 for team in wc_2026_teams}

for _ in range(N_SIMS):
    winner = simulate_tournament(wc_2026_teams, team_elos)
    win_counts[winner] += 1

win_probs = pd.Series(win_counts).sort_values(ascending=False) / N_SIMS

print(f"2026 World Cup Win Probabilities ({N_SIMS:,} simulations)")
print()
for team, prob in win_probs.items():
    bar = '█' * int(prob * 100)
    print(f"  {team:<25} {prob:5.1%}  {bar}")

In [ ]:
# Visualize top 15 contenders
top15 = win_probs.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top15.index[::-1], top15.values[::-1], color='steelblue', edgecolor='white')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_title('2026 FIFA World Cup Win Probabilities (Top 15)', fontsize=14)
ax.set_xlabel('Probability of Winning the Tournament')

for bar, val in zip(bars, top15.values[::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Head-to-Head Predictor

Use the model to predict any matchup on demand.

In [ ]:
def predict_match(team_a, team_b, elos=elo_ratings):
    elo_a = elos.get(team_a, 1500)
    elo_b = elos.get(team_b, 1500)
    elo_diff = elo_a - elo_b
    prob_a = model.predict_proba([[elo_diff, 0]])[0][1]
    prob_b = 1 - prob_a
    print(f"\n{team_a} vs {team_b}")
    print(f"  Elo: {int(elo_a)} vs {int(elo_b)}")
    print(f"  {team_a} win probability:  {prob_a:.1%}")
    print(f"  {team_b} win probability: {prob_b:.1%}")

# Example matchups
predict_match('Brazil', 'France')
predict_match('Argentina', 'England')
predict_match('United States', 'Mexico')

---

## Summary

**Model performance:**
- ROC-AUC: *(fill in after running)*
- Brier Score: *(fill in after running)*

**2026 World Cup favourite:** *(fill in after running)*

**Key findings:**
- Elo rating difference is a strong single predictor of match outcomes
- The model uses time-based validation (trained pre-2018, tested 2018–present) to avoid look-ahead bias
- Win probabilities are well-calibrated: heavy favourites win, but upsets are built into the simulation

**Limitations:**
- Model does not account for injuries, squad depth, or recent form beyond what Elo captures
- 2026 bracket draw is random in simulation; actual group assignments will shift probabilities
- Draws are excluded from the binary model — a future version would use multinomial logistic regression for win/draw/loss